In [13]:
import os

In [12]:
%pwd

'c:\\Users\\sagal\\Desktop\\Let us build\\emotion_detection\\research\\03_inference_trials'

In [14]:
os.chdir('../../')

In [15]:
%pwd

'c:\\Users\\sagal\\Desktop\\Let us build\\emotion_detection'

In [16]:
import pandas as pd
import torch
from torchvision import models
from torchvision.models import resnet18
import torch.nn as nn
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F
import os
import cv2
from PIL import Image
from pathlib import Path
from emotion_detection.config.configuration import configurationManager


## Step 1: Load model

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [18]:
# Load the initial model
NUM_CLASSES = 7

model = resnet18(weights=None)
model.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(model.fc.in_features, 7))

In [19]:
#first import config manager to read config.yaml
config_manager = configurationManager()
config = config_manager.get_model_evaluation_config()

MODEL_PATH = Path(config.model_path)

[2026-07-31 20:54:43,810: INFO: common: YAML file loaded successfully from: config\config.yaml]
[2026-07-31 20:54:43,818: INFO: common: YAML file loaded successfully from: params.yaml]
[2026-07-31 20:54:43,821: INFO: common: created directory at artifacts]
[2026-07-31 20:54:43,825: INFO: common: created directory at artifacts/model_evaluation]


In [21]:
#fix the model path from local device based to config based
MODEL_PATH = Path(config.model_path)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Step 2 : inference preprocessing

In [23]:
# Introduce inference tansform, same as during training
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [25]:
# Get test image
#image = Image.open(r"S:\Projects\emotion_detection\artifacts\data_ingestion\Organized\train\happy\train_00036_aligned.jpg")


#local hardcoding and the image name can change so, the fix here would be to choose the best available image dynamically
#instead of hardcoding and we can use pathlib as we have used it earlier

In [26]:
#automatically pick the first image inside the happy folder
happy_folder = Path("artifacts/data_ingestion/Organized/train/happy")
test_image_path = next(happy_folder.glob("*.jpg"))

image = Image.open(test_image_path)
print(f"Loaded image: {test_image_path}")

Loaded image: artifacts\data_ingestion\Organized\train\happy\train_09748_aligned.jpg


In [27]:
# apply transform
image_tensor = inference_transform(image)

In [28]:
print(image_tensor.shape)

torch.Size([3, 224, 224])


In [29]:
# add batch
image_tensor = image_tensor.unsqueeze(0)

In [30]:
print(image_tensor.shape)

torch.Size([1, 3, 224, 224])


In [31]:
image_tensor = image_tensor.to(device)

## Step 3: single image prediction

In [32]:
with torch.no_grad():
    outputs = model(image_tensor)

print(outputs.shape)    

torch.Size([1, 7])


In [33]:
# Convert logits into probabilities
probabilities = F.softmax(outputs, dim=1)
print(probabilities)

tensor([[0.1754, 0.0398, 0.0081, 0.1786, 0.0479, 0.3462, 0.2040]],
       device='cuda:0')


In [34]:
# Get highest probability
confidence, predicted = torch.max(probabilities, dim=1)

In [35]:
# Label map
class_names = [
    'angry', 
    'disgust', 
    'fear', 
    'happy', 
    'neutral', 
    'sad', 
    'surprise'
]

In [36]:
emotion = class_names[predicted.item()]

print(f"Prediction : {emotion}")
print(f"Confidence : {confidence.item()*100:.2f}%")

Prediction : sad
Confidence : 34.62%


In [37]:
# helper function for testing
class_names = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise"
]


def predict_emotion(
    model,
    transform,
    device,
    image_path=None,
    frame=None
):
    """
    Predict emotion from a single image.

    Args:
        image_path (str): Path to image.
        model: Trained PyTorch model.
        transform: Inference transform.
        device: cuda or cpu.

    Returns:
        predicted_label, confidence
    """

    # Load image
    if image_path is not None:

        image = Image.open(image_path).convert("RGB")

    elif frame is not None:

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(image)

    else:
        raise ValueError("Provide either image_path or frame.")

    # Preprocess
    image = transform(image)

    # Add batch dimension
    image = image.unsqueeze(0).to(device)

    # Prediction
    model.eval()
    with torch.no_grad():
        outputs = model(image)

        probabilities = F.softmax(outputs, dim=1)

        confidence, predicted = torch.max(probabilities, dim=1)

    predicted_label = class_names[predicted.item()]
    confidence = confidence.item() * 100

    # Convert probabilities into a dictionary
    all_probabilities = {
        label: prob * 100
        for label, prob in zip(
            class_names,
            probabilities.squeeze().cpu().numpy()
        )
    }

    return predicted_label, confidence, all_probabilities

In [38]:
prediction, confidence, probabilities = predict_emotion(
    image_path=r"S:\Projects\emotion_detection\artifacts\data_ingestion\Organized\train\sad\train_00692_aligned.jpg",
    model=model,
    transform=inference_transform,
    device=device
)

print(f"\nPrediction : {prediction}")
print(f"Confidence : {confidence:.2f}%\n")

print("Class Probabilities:")
for emotion, prob in probabilities.items():
    print(f"{emotion:<10}: {prob:.2f}%")

FileNotFoundError: [Errno 2] No such file or directory: 'S:\\Projects\\emotion_detection\\artifacts\\data_ingestion\\Organized\\train\\sad\\train_00692_aligned.jpg'

In [39]:
#we are going to do the same here as well, remove hardcoding, use referencing and use the first image available from the directory

In [40]:

#define directry from where you want the image
sad_folder = Path("artifacts/data_ingestion/Organized/train/sad")

#first available image dynamically
test_image_path = next(sad_folder.glob("*.jpg"))

print(f"Testing with image: {test_image_path}")

#pass the path to predict_emotion
prediction, confidence, probabilities = predict_emotion(
    image_path=str(test_image_path),
    model=model,
    transform=inference_transform,
    device=device
)

print(f"\nPrediction : {prediction}")
print(f"Confidence : {confidence:.2f}%\n")

print("Class Probabilities:")
for emotion, prob in probabilities.items():
    print(f"{emotion:<10}: {prob:.2f}%")

Testing with image: artifacts\data_ingestion\Organized\train\sad\train_00001_aligned.jpg

Prediction : happy
Confidence : 60.31%

Class Probabilities:
angry     : 4.25%
disgust   : 25.29%
fear      : 0.00%
happy     : 60.31%
neutral   : 0.00%
sad       : 10.14%
surprise  : 0.00%


### Batch inference

In [41]:
def batch_predict(folder_path, model, transform, device):
    """
    Predict emotions for all images in a folder.

    Args:
        folder_path (str): Folder containing images.
        model: Trained PyTorch model.
        transform: Inference transform.
        device: cpu or cuda.

    Returns:
        List of prediction results.
    """

    results = []

    supported_extensions = (".jpg", ".jpeg", ".png", ".bmp") 

    for image_name in os.listdir(folder_path):

        if not image_name.lower().endswith(supported_extensions):
            continue

        image_path = os.path.join(folder_path, image_name)

        prediction, confidence, probabilities = predict_emotion(
            image_path=image_path,
            model=model,
            transform=transform,
            device=device
        )

        results.append({
            "image": image_name,
            "prediction": prediction,
            "confidence": confidence,
            "probabilities": probabilities
        })

    return results

In [44]:
test_folder = Path('test_images')

results = batch_predict(
    folder_path=str(test_folder),
    model=model,
    transform=inference_transform,
    device=device
)

In [45]:
for result in results:

    print("-" * 50)

    print(f"Image      : {result['image']}")
    print(f"Prediction : {result['prediction']}")
    print(f"Confidence : {result['confidence']:.2f}%")

    print("\nClass Probabilities:")

    for emotion, prob in result["probabilities"].items():
        print(f"{emotion:<10}: {prob:.2f}%")

--------------------------------------------------
Image      : happy_1.jpg
Prediction : sad
Confidence : 75.04%

Class Probabilities:
angry     : 0.03%
disgust   : 0.15%
fear      : 0.04%
happy     : 1.93%
neutral   : 0.46%
sad       : 75.04%
surprise  : 22.36%
--------------------------------------------------
Image      : happy_2.jpg
Prediction : happy
Confidence : 52.69%

Class Probabilities:
angry     : 10.67%
disgust   : 0.14%
fear      : 0.02%
happy     : 52.69%
neutral   : 0.01%
sad       : 0.72%
surprise  : 35.75%
--------------------------------------------------
Image      : neutral_1.jpg
Prediction : happy
Confidence : 58.25%

Class Probabilities:
angry     : 0.80%
disgust   : 5.42%
fear      : 0.02%
happy     : 58.25%
neutral   : 6.24%
sad       : 17.86%
surprise  : 11.41%
--------------------------------------------------
Image      : neutral_2.jpg
Prediction : surprise
Confidence : 50.54%

Class Probabilities:
angry     : 0.27%
disgust   : 0.40%
fear      : 0.19%
happy  

In [47]:
# Path relative to project root
test_sad_folder = Path("artifacts/data_ingestion/Organized/test/sad")
test_image_path = next(test_sad_folder.glob("*.jpg"))

prediction, confidence, probabilities = predict_emotion(
    image_path=str(test_image_path),
    model=model,
    transform=inference_transform,
    device=device
)

### Real time inference

In [48]:
# Load pretrained face detector
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

In [ ]:
# Replace webcam loop with face detector
cap = cv2.VideoCapture(0)

face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(100, 100)
    )

    for (x, y, w, h) in faces:

        face = frame[y:y+h, x:x+w]

        prediction, confidence, _ = predict_emotion(
            model=model,
            transform=inference_transform,
            device=device,
            frame=face
        )

        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        label = f"{prediction} ({confidence:.1f}%)"

        cv2.putText(
            frame,
            label,
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    cv2.imshow("Emotion Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

: 